# Querying the lakehouse from Python

Connects to Trino on port 8080. Works from VS Code (local `.venv`) or from the Jupyter container at http://localhost:8888.

In [1]:
import os

import pandas as pd
import trino

conn = trino.dbapi.connect(host=os.getenv("TRINO_HOST", "localhost"), port=8080, user="amanda", catalog="iceberg", schema="raw")


def sql(query: str) -> pd.DataFrame:
    cur = conn.cursor()
    cur.execute(query)
    return pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])

In [2]:
sql("SHOW TABLES")

,Table
0,bluebikes_trips
1,boston_bike_counts_daily
2,boston_bike_counts_hourly
3,boston_crashes
4,nyc_bike_routes
5,nyc_crashes


## Washington St near Hyde Park Ave, summer counts

In [3]:
sql("""
SELECT year,
       round(avg(CAST(day_bikes AS integer))) AS avg_bikes,
       round(avg(CAST(day_veh AS integer))) AS avg_vehicles
FROM boston_bike_counts_daily
WHERE location_name = 'Washington Street north of Bexley Road' AND season = 'Summer'
GROUP BY year
ORDER BY year
""")

,year,avg_bikes,avg_vehicles
0,2016,361.0,17414.0
1,2018,331.0,16913.0
2,2019,380.0,17153.0
3,2020,310.0,13401.0
4,2021,334.0,16910.0
5,2022,388.0,17219.0
6,2023,328.0,16798.0
7,2024,468.0,17066.0
8,2025,547.0,16401.0
9,2026,574.0,15826.0


## NYC protected bike lane segments by install year

In [4]:
sql("""
SELECT year(CAST(substr(instdate, 1, 10) AS date)) AS install_year, count(*) AS protected_segments
FROM nyc_bike_routes
WHERE 'Protected' IN (ft_facilit, tf_facilit) AND instdate IS NOT NULL
GROUP BY 1
ORDER BY 1 DESC
LIMIT 10
""")

,install_year,protected_segments
0,2026,158
1,2025,210
2,2024,570
3,2023,743
4,2022,450
5,2021,558
6,2020,518
7,2019,426
8,2018,465
9,2017,434


## Live stations (Postgres) next to a year of trips (Iceberg)

Run `uv run bikelake stations` or materialize `station_status` in Dagster first.

In [5]:
sql(open("../sql/02_live_and_history.sql").read().split(";")[0])

,name,bikes_available,docks_available,usual_departures_this_hour
0,Deerfield St at Commonwealth Ave,35,6,2.9
1,MIT Vassar St,34,19,2.8
2,Lansdowne T Stop,40,11,2.3
3,Kenmore Square,7,18,2.1
4,Landmark Center - Brookline Ave at Park Dr,8,6,1.5
5,Silber Way,1,17,1.2
6,Park Dr at Buswell St,14,0,1.1
7,Burlington Ave at Brookline Ave,5,10,0.8
8,Commonwealth Ave at Granby St,2,14,0.7
9,St Mary's,14,1,0.7


## Snapshots and time travel

Every load is a snapshot. Read the table as it was at the first load.

In [6]:
snapshots = sql('SELECT snapshot_id, operation, committed_at FROM "boston_bike_counts_daily$snapshots" ORDER BY committed_at')
snapshots

,snapshot_id,operation,committed_at
0,2252313084999985002,append,2026-11-07 02:23:16.181+00:00
1,5451624008893660454,delete,2026-11-07 02:37:26.102+00:00
2,4162040988446579316,append,2026-11-07 02:37:26.190+00:00
3,3259083012319621024,delete,2026-11-07 02:52:57.611+00:00
4,1013948077622893431,append,2026-11-07 02:52:57.661+00:00
5,6002040932842698620,delete,2026-11-07 03:40:18.500+00:00
6,7627512119050744363,append,2026-11-07 03:40:18.547+00:00
7,7501821893914079195,delete,2026-11-07 03:41:13.423+00:00
8,3074424126364150938,append,2026-11-07 03:41:13.468+00:00


In [7]:
first = snapshots.snapshot_id.iloc[0]
sql(f"SELECT count(*) AS rows_at_first_load FROM boston_bike_counts_daily FOR VERSION AS OF {first}")

,rows_at_first_load
0,2952
